In [6]:
import os
from typing import Annotated, List, Literal, Optional

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from enum import Enum


In [2]:
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=512,
)


In [7]:
class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL =  "neutral"

In [19]:
# schema
class Review(BaseModel):
    key_themes: list[str] = Field(...,
        description="Identify the major topics and points discussed in the review."
    )
    summary: str = Field(...,
        description="Summarize the reviewer's main opinion and key observations."
    )
    sentiment: Sentiment = Field(...,
        description="Classify the overall sentiment as positive, negative, or neutral."
    )
    pros: list[str] | None = Field(
        default=None,
        description="List explicitly mentioned advantages, or None if there are none."
    )
    cons: list[str] | None = Field(
        default=None,
        description="List explicitly mentioned disadvantages, or None if there are none."
    )
    name: str | None = Field(
        default=None,
        description="Extract the reviewer's name if explicitly provided; otherwise None."
    )

In [9]:
stuctured_model = model.with_structured_output(Review)

In [10]:
review_text = """
I purchased these wireless headphones two weeks ago. The sound quality
is excellent, and the battery lasts almost 30 hours. They are comfortable
to wear for long periods. However, the microphone quality is poor during
calls, and the plastic build feels cheap. Overall, I am satisfied with
the purchase. — Rahul Sharma
"""


In [13]:
result = stuctured_model.invoke(review_text)

In [16]:
dic_obj = result.model_dump()

In [17]:
for key, value in dic_obj.items():
    print(f"{key} ----> {value}")


key_themes ----> ['sound quality', 'battery life', 'comfort', 'microphone quality', 'build quality']
summary ----> The reviewer finds the wireless headphones to have excellent sound quality, impressive battery life, and comfort for long usage, but notes poor microphone quality and cheap-feeling build.
sentiment ----> Sentiment.POSITIVE
pros ----> ['excellent sound quality', 'long battery life (almost 30 hours)', 'comfortable for long use']
cons ----> ['poor microphone quality during calls', 'cheap plastic build']
name ----> Rahul Sharma
